In [ ]:
# Import necessary libraries
import numpy as np
import gym
import random
import matplotlib.pyplot as plt

# Create a simple environment
env = gym.make('CartPole-v1')

# Initialize Q-table
n_actions = env.action_space.n
n_states = np.prod(env.observation_space.shape)
q_table = np.zeros((n_states, n_actions))

# Define hyperparameters
alpha = 0.1  # Learning rate
gamma = 0.99  # Discount factor
epsilon = 0.1  # Exploration rate
episodes = 1000

# Discretize the state space
def discretize_state(state):
    return tuple(np.digitize(state, bins=np.linspace(-1.2, 1.2, num=10)))

# Training loop
for episode in range(episodes):
    state = env.reset()
    state = discretize_state(state)
    
    done = False
    total_reward = 0
    
    while not done:
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()  # Explore
        else:
            action = np.argmax(q_table[state])  # Exploit
            
        next_state, reward, done, _ = env.step(action)
        next_state = discretize_state(next_state)
        
        # Update Q-table using Q-learning formula
        q_table[state][action] = q_table[state][action] + alpha * (reward + gamma * np.max(q_table[next_state]) - q_table[state][action])
        
        state = next_state
        total_reward += reward
    
    if episode % 100 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward}")

# Save the Q-table
np.save('q_table.npy', q_table)

# Example testing the trained agent
state = env.reset()
state = discretize_state(state)
done = False
while not done:
    action = np.argmax(q_table[state])  # Choose action based on learned policy
    state, reward, done, _ = env.step(action)
    env.render()  # Show the environment
